# Gradient Boosting

## Pourquoi  ?

Gradient Boosting est particulièrement adapté à notre problème car
il construit ses arbres **en séquence** — chaque arbre corrige les
erreurs du précédent.

C'est très utile ici car certains Légendaires sont "atypiques".
Cosmog par exemple a des stats très faibles pour un Légendaire,
Shaymin ressemble beaucoup à un Pokémon normal...
Ces cas difficiles vont être progressivement corrigés au fil des arbres.

De plus, comme Random Forest, il gère naturellement les **relations non-linéaires**  et fournit
une **feature importance** pour analyser quelles stats définissent
vraiment un Légendaire.

Enfin il est robuste au déséquilibre des classes grâce au paramètre
`scale_pos_weight` qui donne plus d'importance aux Légendaires
pendant l'entraînement.

## Différence entre Random Forest et Gradient Boosting
- **Random Forest**  : construit plein d'arbres en même temps, chacun sur un échantillon aléatoire différent, puis ils votent ensemble
- **Gradient Boosting** :  construit ses arbres un par un, chaque arbre apprend des erreurs du précédent

## 1. Imports et chargement des données



In [ ]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Chargement du DataFrame
df = pd.read_csv("../../data/encoded/pokemon_encoded.csv")

## 2. Séparation des données

In [ ]:
# Séparation des données en caractéristiques (X) et cible (y)
y = df['Legendary']
# La colone cible est "Legendary", on la sépare du reste des données pour l'entraînement du modèle
X = df.drop(columns=['Legendary'])

# Division des données en ensembles d'entraînement et de test
# On utilise une stratification pour s'assurer que la proportion de Pokémon légendaires et normaux est similaire dans les deux ensembles
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 3. Entrainement des données

### 3.1 On cherche les meilleurs parametre avec ``GridSearchCV``

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Meilleurs paramètres :", grid_search.best_params_)

``learning_rate`` : C'est la vitesse à laquelle chaque arbre corrige les erreurs du précédent
- Si trop élevé (0.2) : il corrige trop vite et fait des erreurs
- Si trop faible (0.01) : il apprend très lentement mais plus précisément

Le GridSearch va trouver le meilleur équilibre automatiquement 

### 3.2 On entraine les données

In [ ]:
# Entraînement du modèle avec les meilleurs paramètres
gb_best = GradientBoostingClassifier(
    **grid_search.best_params_,
    random_state=42
)

gb_best.fit(X_train, y_train)
y_pred = gb_best.predict(X_test)

## 4. Evaluation

### 4.1 Matrice de confusion

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Normal', 'Légendaire'],
            yticklabels=['Normal', 'Légendaire'])
plt.title("Matrice de confusion — Gradient Boosting")
plt.ylabel("Réel")
plt.xlabel("Prédit")
plt.show()

# Rapport de classification
print(classification_report(y_test, y_pred, target_names=['Normal', 'Légendaire']))
print("Meilleur score F1 (cross validation) :", grid_search.best_score_)

...

### 4.2 Colonnes importantes

In [ ]:
importances = pd.Series(gb_best.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

importances.plot(kind='barh', color='#e91e8c', figsize=(10, 8))
plt.title("Importance des variables — Gradient Boosting")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()